# Multiverse Hybrid v3.0 — Stage 0 PRICE Bulk 2000 v2

v1の実行ログが `CalledProcessError` に隠れたため、実行診断を強化した版です。科学仕様は変更していません。

**iPhoneでは『ランタイム → すべてのセルを実行』だけで構いません。**


In [ ]:
!pip -q install lxml

from google.colab import drive, files
from pathlib import Path
import subprocess, shutil, json, os, textwrap

drive.mount('/content/drive')
REPO=Path('/content/multiverse-research-stage0-price-v2')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.check_call(['git','clone','--depth','1','https://github.com/fufufu1116/multiverse-research.git',str(REPO)])

EXPECTED={
 'v3/historical_all_market/stage0_price_bulk_runner_v2.py':'ff2a045c9f7aa893f9bd119a52d27e0899298340',
 'v3/historical_all_market/kdreams_price_catalog_recovery_v1.py':'f94a08a3ea7c0a4f110dc0df82433eecc25b0cf8',
 'v3/historical_all_market/governance/INDEPENDENT_GEMINI_STAGE0_FINAL_APPROVE_RECEIPT_v1.json':'8643684cf7bf0165968ae667e17e546936a3611d',
}
for rel,exp in EXPECTED.items():
    obs=subprocess.check_output(['git','-C',str(REPO),'hash-object',rel],text=True).strip()
    if obs!=exp: raise RuntimeError(f'FAIL-CLOSED Git blob mismatch {rel}: {obs} != {exp}')
print('✅ EXACT CODE / APPROVAL BINDINGS PASS')

# Local Drive input sanity before launching the bulk subprocess.
MY=Path('/content/drive/MyDrive')
U=MY/'MULTIVERSE_DEV2000_UNIVERSE_RECOVERY'/'DEV2000_UNIVERSE_v1.csv'
P=MY/'MULTIVERSE_DEV2000_RESULT_COLLECTION_v3_HARDENED'/'DEV2000_RESULT_PROVENANCE_v3.jsonl'
R=MY/'MULTIVERSE_DEV2000_RESULT_COLLECTION_v3_HARDENED'/'RAW_RESULT_QUARANTINE'
print('INPUT PATHS:')
print(' universe:',U, U.exists())
print(' provenance:',P, P.exists())
print(' raw_dir:',R, R.exists())
if not (U.is_file() and P.is_file() and R.is_dir()):
    raise RuntimeError('FAIL-CLOSED required Drive input path missing')

OUT=MY/'MULTIVERSE_ALL_MARKET_STAGE0_PRICE_RECOVERY_v2'
OUT.mkdir(parents=True,exist_ok=True)
LOG=OUT/'STAGE0_PRICE_BULK_RUN_LOG_v2.txt'
runner=REPO/'v3/historical_all_market/stage0_price_bulk_runner_v2.py'
cmd=['python',str(runner),'--mydrive',str(MY),'--repo-root',str(REPO),'--overwrite']
print('▶ Stage0 PRICE-only 2000R bulk start')
proc=subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
LOG.write_text(proc.stdout,encoding='utf-8')
tail='\n'.join(proc.stdout.splitlines()[-160:])
print(tail)
print(f'\nRETURN CODE = {proc.returncode}')
print('FULL LOG:',LOG)

FATAL=OUT/'STAGE0_PRICE_BULK_FATAL_v2.json'
if proc.returncode!=0:
    if FATAL.exists():
        print('\n=== FAIL-CLOSED FATAL RECEIPT ===')
        print(FATAL.read_text(encoding='utf-8'))
        files.download(str(FATAL))
    files.download(str(LOG))
    raise RuntimeError(f'Stage0 PRICE bulk stopped FAIL-CLOSED. returncode={proc.returncode}. Full cause saved in Drive log.')

receipt=OUT/'STAGE0_PRICE_BULK_RECEIPT_v2.json'
quality=OUT/'PRICE_ONLY'/'POST_BULK_PRICE_QUALITY_REPORT_v2.json'
if not receipt.exists() or not quality.exists():
    raise RuntimeError('FAIL-CLOSED success return but receipt/quality missing')
print('\n=== STAGE0 PRICE BULK RECEIPT ===')
print(receipt.read_text(encoding='utf-8'))
print('\n=== POST-BULK QUALITY REPORT ===')
print(quality.read_text(encoding='utf-8'))
files.download(str(receipt))
files.download(str(quality))
files.download(str(LOG))
